# LeetCode 235 — Lowest Common Ancestor of a Binary Search Tree bằng Python

Link đề: https://leetcode.com/problems/lowest-common-ancestor-of-a-binary-search-tree/ · Độ khó: **Medium** · Chủ đề: Tree, Depth-First Search, Binary Search Tree, Binary Tree, Lowest Common Ancestor

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, x):
#         self.val = x
#         self.left = None
#         self.right = None
class Solution:
    def lowestCommonAncestor(self, root: 'TreeNode', p: 'TreeNode', q: 'TreeNode') -> 'TreeNode':
        ...
```

Ghi chú: đề, ví dụ và constraints lấy từ đề gốc; mọi đáp án trong notebook đều được kiểm chứng bằng code và đối chiếu với một lời giải độc lập đi đúng theo **định nghĩa** (xét mọi nút, kiểm tra cả `p` và `q` nằm trong cây con của nút đó, lấy nút sâu nhất).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm **tổ tiên chung thấp nhất (LCA)** của hai nút `p`, `q` trong một **cây tìm kiếm nhị phân (BST)**: nút **thấp nhất** (xa gốc nhất) có **cả `p` và `q` là hậu duệ** của nó (một nút được coi là hậu duệ của chính nó).
2. Biết **tự dựng cây** (`TreeNode`, `build_tree` từ danh sách level-order) và **tìm nút theo giá trị** để làm đầu vào `p`, `q` khi test.
3. Nhận ra **tính chất then chốt của BST**: tại mỗi nút, nếu cả `p` và `q` nhỏ hơn nút thì LCA nằm bên **trái**; cả hai lớn hơn thì nằm bên **phải**; còn lại (hai giá trị nằm hai phía hoặc một trong hai bằng nút) thì **chính nút này** là LCA.
4. Đi từ **đường đi gốc → nút trên cây tổng quát** → **đường đi theo giá trị trong BST** → **đệ quy theo tính chất BST** → **vòng lặp `O(1)` bộ nhớ** (cách nộp bài) → **dùng lời giải cho cây nhị phân tổng quát** để thấy tính chất BST giúp nhanh hơn.
5. Tránh các lỗi kinh điển: kiểm tra điều kiện bằng `or` thay vì `and`, bỏ sót trường hợp nút là tổ tiên của chính mình, giả định `p.val < q.val`, và trả về giá trị thay vì nút.

## 2. Đề bài & ví dụ

Cho một **cây tìm kiếm nhị phân (BST)**, hãy tìm **tổ tiên chung thấp nhất (LCA)** của hai nút cho trước trong BST.

Theo định nghĩa trên Wikipedia: "Tổ tiên chung thấp nhất được định nghĩa giữa hai nút `p` và `q` là nút thấp nhất trong cây `T` có cả `p` và `q` là hậu duệ (ta cho phép một nút là hậu duệ của chính nó)."

**Ví dụ 1**

- Input: `root = [6,2,8,0,4,7,9,null,null,3,5]`, `p = 2`, `q = 8`
- Output: `6`
- Giải thích: LCA của nút `2` và nút `8` là `6`.

**Ví dụ 2**

- Input: `root = [6,2,8,0,4,7,9,null,null,3,5]`, `p = 2`, `q = 4`
- Output: `2`
- Giải thích: LCA của `2` và `4` là `2`, vì một nút có thể là hậu duệ của chính nó.

**Ví dụ 3**

- Input: `root = [2,1]`, `p = 2`, `q = 1`
- Output: `2`

**Ràng buộc:** số nút trong khoảng `[2, 10⁵]`; `-10⁹ <= Node.val <= 10⁹`; mọi `Node.val` đều **khác nhau**; `p != q`; `p` và `q` đều tồn tại trong BST.

In [1]:
from typing import List, Optional
import random
from collections import deque


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    """Dựng cây từ danh sách level-order (None = nút vắng mặt), giống định dạng của LeetCode."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root


def find_node(root, val):
    """Tìm nút có giá trị val (duyệt tổng quát bằng ngăn xếp) để làm đầu vào p, q khi test."""
    stack = [root] if root else []
    while stack:
        node = stack.pop()
        if node.val == val:
            return node
        if node.left:
            stack.append(node.left)
        if node.right:
            stack.append(node.right)
    return None


examples = [
    ([6, 2, 8, 0, 4, 7, 9, None, None, 3, 5], 2, 8, 6),
    ([6, 2, 8, 0, 4, 7, 9, None, None, 3, 5], 2, 4, 2),
    ([2, 1], 2, 1, 2),
]
for values, pv, qv, expected in examples:
    root = build_tree(values)
    p, q = find_node(root, pv), find_node(root, qv)
    print(f"root={values}, p={p.val}, q={q.val}, kỳ vọng {expected}")

root=[6, 2, 8, 0, 4, 7, 9, None, None, 3, 5], p=2, q=8, kỳ vọng 6
root=[6, 2, 8, 0, 4, 7, 9, None, None, 3, 5], p=2, q=4, kỳ vọng 2
root=[2, 1], p=2, q=1, kỳ vọng 2


## 3. Kiến thức nền

### 3.1. Nút cây, `build_tree` và tìm nút theo giá trị

Mỗi nút có `val`, `left`, `right`. Đề cho `p` và `q` là **nút** (đối tượng `TreeNode`), không phải số; khi test ta dựng cây rồi gọi `find_node(root, giá_trị)` để lấy đúng đối tượng nút trong cây. Kết quả trả về cũng là một **nút** (ta so sánh `.val` hoặc so sánh danh tính bằng `is`).

In [2]:
root = build_tree([6, 2, 8, 0, 4, 7, 9, None, None, 3, 5])
p = find_node(root, 2)
print(p.val, p.left.val, p.right.val)           # 2 0 4
print(find_node(root, 99))                       # None (không có)
print(find_node(root, 2) is p)                   # True: cùng một đối tượng nút

2 0 4
None
True


### 3.2. Tính chất BST: đi tìm một giá trị chỉ cần đi một nhánh

Trong BST, nếu giá trị cần tìm nhỏ hơn nút hiện tại thì nó (nếu có) nằm ở cây con **trái**; lớn hơn thì ở cây con **phải**. Vì vậy việc đi từ gốc xuống một nút mất tối đa `h` bước (`h` là độ cao cây) thay vì duyệt cả cây.

In [3]:
def bst_path(root, target_val):
    """Đường đi từ gốc đến nút có giá trị target_val trong BST (danh sách giá trị)."""
    path, cur = [], root
    while cur:
        path.append(cur.val)
        if target_val < cur.val:
            cur = cur.left
        elif target_val > cur.val:
            cur = cur.right
        else:
            return path
    return None


root = build_tree([6, 2, 8, 0, 4, 7, 9, None, None, 3, 5])
print(bst_path(root, 3))      # [6, 2, 4, 3]
print(bst_path(root, 8))      # [6, 8]
print(bst_path(root, 1))      # None

[6, 2, 4, 3]
[6, 8]
None


### 3.3. Hiểu định nghĩa LCA bằng "đường đi từ gốc"

Mọi **tổ tiên** của một nút nằm trên đường đi từ gốc tới nút đó (kể cả chính nó). Tổ tiên **chung** của `p` và `q` là các nút nằm trên **cả hai** đường đi, và đó là phần **tiền tố chung** của hai đường. LCA là nút **cuối cùng** (sâu nhất) của tiền tố chung.

In [4]:
root = build_tree([6, 2, 8, 0, 4, 7, 9, None, None, 3, 5])
path_p = bst_path(root, 3)     # [6, 2, 4, 3]
path_q = bst_path(root, 5)     # [6, 2, 4, 5]
common = []
for a, b in zip(path_p, path_q):
    if a != b:
        break
    common.append(a)
print(path_p, path_q)
print("tiền tố chung:", common, "-> LCA =", common[-1])      # 4

[6, 2, 4, 3] [6, 2, 4, 5]
tiền tố chung: [6, 2, 4] -> LCA = 4


### 3.4. `min`, `max` và so sánh theo giá trị

Ta sẽ cần hai giá trị đã sắp thứ tự: `lo = min(p.val, q.val)`, `hi = max(p.val, q.val)`. Khi đó nút `v` nằm "giữa" khi `lo <= v.val <= hi`. Không được giả định `p.val < q.val`: đề không đảm bảo thứ tự của `p` và `q`.

In [5]:
for pv, qv in [(2, 8), (8, 2), (4, 2)]:
    lo, hi = min(pv, qv), max(pv, qv)
    print(f"p={pv}, q={qv} -> lo={lo}, hi={hi}")

p=2, q=8 -> lo=2, hi=8
p=8, q=2 -> lo=2, hi=8
p=4, q=2 -> lo=2, hi=4


### 3.5. Giới hạn đệ quy và vòng lặp `while`

Cây BST có thể **lệch** (chuỗi `10⁵` nút) nên đệ quy dễ gây `RecursionError` (Python mặc định khoảng 1000 tầng). Vòng lặp `while` không có giới hạn này và còn dùng `O(1)` bộ nhớ: ta sẽ ưu tiên nó cho cách nộp bài.

In [6]:
import sys
print("giới hạn đệ quy mặc định:", sys.getrecursionlimit())


def depth_recursive(node):
    return 0 if node is None else 1 + max(depth_recursive(node.left), depth_recursive(node.right))


chain = TreeNode(0)
cur = chain
for v in range(1, 2000):                      # cây lệch phải sâu 2000
    cur.right = TreeNode(v)
    cur = cur.right
try:
    depth_recursive(chain)
except RecursionError:
    print("RecursionError: đệ quy quá sâu trên cây lệch")

giới hạn đệ quy mặc định: 1000
RecursionError: đệ quy quá sâu trên cây lệch


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Cây ví dụ 1:

```
          6
        /   \
       2     8
      / \   / \
     0   4 7   9
        / \
       3   5
```

**`p = 2`, `q = 8`:** từ gốc `6`: `2 < 6` và `8 > 6` nên hai nút nằm **hai phía** của `6`. Không thể có nút nào sâu hơn `6` chứa cả hai, vậy LCA là `6`.

**`p = 2`, `q = 4`:** từ gốc `6`: cả `2` và `4` đều nhỏ hơn `6` nên xuống con trái `2`. Tại `2`: `p = 2` bằng giá trị nút nên LCA là chính `2` (nút là hậu duệ của chính mình, `4` nằm trong cây con phải của `2`).

**`p = 3`, `q = 5`:** `6` (cả hai nhỏ hơn: trái) → `2` (cả hai lớn hơn: phải) → `4` (`3 < 4 < 5`: hai phía) → LCA là `4`.

**Quy tắc:** đi xuống chừng nào `p` và `q` còn **cùng một phía** của nút hiện tại; nút đầu tiên mà chúng **tách ra** (hoặc một trong hai chính là nút đó) là LCA.

In [7]:
root = build_tree([6, 2, 8, 0, 4, 7, 9, None, None, 3, 5])
for pv, qv in [(2, 8), (2, 4), (3, 5)]:
    lo, hi = min(pv, qv), max(pv, qv)
    cur = root
    walk = []
    while True:
        walk.append(cur.val)
        if cur.val < lo:
            cur = cur.right
        elif cur.val > hi:
            cur = cur.left
        else:
            break
    print(f"p={pv}, q={qv}: đi qua {walk} -> LCA = {cur.val}")

p=2, q=8: đi qua [6] -> LCA = 6
p=2, q=4: đi qua [6, 2] -> LCA = 2
p=3, q=5: đi qua [6, 2, 4] -> LCA = 4


## 5. Cách 1 — Brute Force / baseline: đường đi gốc → nút trên cây tổng quát

**Ý tưởng.** Không dùng tính chất BST: tìm **đường đi** từ gốc tới `p` và tới `q` bằng duyệt tổng quát (ngăn xếp + bản đồ cha), rồi lấy **nút cuối của tiền tố chung** của hai đường đi.

In [8]:
def path_to(root, target):
    """Đường đi gốc -> target (danh sách nút), duyệt tổng quát bằng ngăn xếp và bản đồ cha."""
    parent = {root: None}
    stack = [root]
    while stack:
        node = stack.pop()
        if node is target:
            break
        for child in (node.left, node.right):
            if child:
                parent[child] = node
                stack.append(child)
    path = []
    cur = target
    while cur is not None:
        path.append(cur)
        cur = parent[cur]
    return path[::-1]


def lca_paths_general(root, p, q):
    path_p, path_q = path_to(root, p), path_to(root, q)
    lca = None
    for a, b in zip(path_p, path_q):
        if a is not b:
            break
        lca = a                                  # nút cuối cùng của tiền tố chung
    return lca


for values, pv, qv, expected in examples:
    root = build_tree(values)
    got = lca_paths_general(root, find_node(root, pv), find_node(root, qv))
    print(f"root={values}, p={pv}, q={qv} -> {got.val} (kỳ vọng {expected})")

root=[6, 2, 8, 0, 4, 7, 9, None, None, 3, 5], p=2, q=8 -> 6 (kỳ vọng 6)
root=[6, 2, 8, 0, 4, 7, 9, None, None, 3, 5], p=2, q=4 -> 2 (kỳ vọng 2)
root=[2, 1], p=2, q=1 -> 2 (kỳ vọng 2)


**Độ phức tạp (Cách 1).** Gọi `n` là số nút. Mỗi lần tìm đường đi duyệt tối đa `n` nút: **Time `O(n)`**, **Space `O(n)`** (bản đồ cha + ngăn xếp).

**Nút thắt.** Cách này đúng cho **mọi** cây nhị phân nhưng phải duyệt cả cây. Với BST, giá trị cho ta biết nên đi **nhánh nào**, nên đường đi dài tối đa `h` bước.

## 6. Ý tưởng thuật toán / bước đột phá

**Tại mỗi nút `v` của BST, so sánh `v.val` với `lo = min(p.val, q.val)` và `hi = max(p.val, q.val)`:**

- `v.val < lo`: cả `p` và `q` đều **lớn hơn** `v`, nên cả hai nằm ở cây con phải. LCA (nếu ở trong cây con của `v`) nằm ở **phải**.
- `v.val > hi`: cả hai **nhỏ hơn** `v`: LCA ở **trái**.
- còn lại `lo ≤ v.val ≤ hi`: hoặc `p` và `q` nằm ở hai phía của `v` (`lo < v.val < hi`), hoặc một trong hai chính là `v` (`v.val == lo` hoặc `hi`). Trong cả hai trường hợp, **`v` là LCA**: nó là hậu duệ chung đầu tiên khi đi từ gốc xuống, và không nút nào sâu hơn chứa cả hai.

Vì chỉ đi xuống **một nhánh** mỗi bước và mỗi bước chỉ làm vài phép so sánh, thuật toán chạy trong `O(h)` thời gian, và viết bằng vòng lặp thì dùng `O(1)` bộ nhớ.

## 7. Các cách tiếp theo

### Cách 2 — Đường đi theo giá trị trong BST (`O(h)`) rồi tiền tố chung

**Giải thích.** Dùng `bst_path` (đi xuống theo giá trị) để lấy đường đi tới `p` và tới `q` chỉ mất `O(h)` mỗi đường; lấy nút cuối của tiền tố chung như Cách 1.

In [9]:
def node_path_bst(root, target):
    path, cur = [], root
    while cur:
        path.append(cur)
        if target.val < cur.val:
            cur = cur.left
        elif target.val > cur.val:
            cur = cur.right
        else:
            return path
    return path


def lca_paths_bst(root, p, q):
    path_p, path_q = node_path_bst(root, p), node_path_bst(root, q)
    lca = None
    for a, b in zip(path_p, path_q):
        if a is not b:
            break
        lca = a
    return lca


for values, pv, qv, expected in examples:
    root = build_tree(values)
    got = lca_paths_bst(root, find_node(root, pv), find_node(root, qv))
    print(f"root={values}, p={pv}, q={qv} -> {got.val} (kỳ vọng {expected})")

root=[6, 2, 8, 0, 4, 7, 9, None, None, 3, 5], p=2, q=8 -> 6 (kỳ vọng 6)
root=[6, 2, 8, 0, 4, 7, 9, None, None, 3, 5], p=2, q=4 -> 2 (kỳ vọng 2)
root=[2, 1], p=2, q=1 -> 2 (kỳ vọng 2)


**Độ phức tạp Cách 2:** **Time `O(h)`**, **Space `O(h)`** (hai danh sách đường đi). Nhanh hơn Cách 1 nhưng còn dùng bộ nhớ.

### Cách 3 — Đệ quy theo tính chất BST

**Giải thích.** `lca(v)`: nếu `v.val < lo` thì `lca(v.right)`; nếu `v.val > hi` thì `lca(v.left)`; ngược lại trả `v`. Gọn nhưng dùng ngăn xếp đệ quy `O(h)`.

In [10]:
def lca_recursive(root, p, q):
    lo, hi = min(p.val, q.val), max(p.val, q.val)
    if root.val < lo:
        return lca_recursive(root.right, p, q)      # cả hai ở bên phải
    if root.val > hi:
        return lca_recursive(root.left, p, q)       # cả hai ở bên trái
    return root                                     # tách ra (hoặc trùng nút): đây là LCA


for values, pv, qv, expected in examples:
    root = build_tree(values)
    got = lca_recursive(root, find_node(root, pv), find_node(root, qv))
    print(f"root={values}, p={pv}, q={qv} -> {got.val} (kỳ vọng {expected})")

root=[6, 2, 8, 0, 4, 7, 9, None, None, 3, 5], p=2, q=8 -> 6 (kỳ vọng 6)
root=[6, 2, 8, 0, 4, 7, 9, None, None, 3, 5], p=2, q=4 -> 2 (kỳ vọng 2)
root=[2, 1], p=2, q=1 -> 2 (kỳ vọng 2)


**Độ phức tạp Cách 3:** **Time `O(h)`**, **Space `O(h)`** (ngăn xếp đệ quy); trên cây lệch `h = n` có thể gây `RecursionError`.

### Cách 4 — Vòng lặp `while` (cách của lời giải tham khảo, `O(1)` bộ nhớ)

**Giải thích.** Cùng ý với Cách 3 nhưng thay lời gọi đệ quy bằng cập nhật biến `root`. Không dùng thêm bộ nhớ và không có giới hạn độ sâu.

In [11]:
def lca_iterative(root, p, q):
    lo, hi = min(p.val, q.val), max(p.val, q.val)
    while root:
        if root.val < lo:
            root = root.right
        elif root.val > hi:
            root = root.left
        else:
            return root
    return None


for values, pv, qv, expected in examples:
    root = build_tree(values)
    got = lca_iterative(root, find_node(root, pv), find_node(root, qv))
    print(f"root={values}, p={pv}, q={qv} -> {got.val} (kỳ vọng {expected})")

root=[6, 2, 8, 0, 4, 7, 9, None, None, 3, 5], p=2, q=8 -> 6 (kỳ vọng 6)
root=[6, 2, 8, 0, 4, 7, 9, None, None, 3, 5], p=2, q=4 -> 2 (kỳ vọng 2)
root=[2, 1], p=2, q=1 -> 2 (kỳ vọng 2)


In [12]:
# Code nộp LeetCode cho Cách 4
class Solution:
    def lowestCommonAncestor(self, root: 'TreeNode', p: 'TreeNode', q: 'TreeNode') -> 'TreeNode':
        while True:
            if root.val < min(p.val, q.val):
                root = root.right
            elif root.val > max(p.val, q.val):
                root = root.left
            else:
                return root

**Độ phức tạp Cách 4:** **Time `O(h)`** (tệ nhất `O(n)` với cây lệch, trung bình `O(log n)` với cây cân bằng), **Space `O(1)`**.

### Cách 5 — Lời giải cho cây nhị phân tổng quát (không dùng tính chất BST)

**Giải thích.** Đệ quy hậu thứ tự: nếu nút rỗng hoặc là `p` hoặc `q` thì trả về chính nó; gọi đệ quy hai bên; nếu cả hai bên đều trả về khác rỗng thì nút hiện tại là LCA, ngược lại trả về bên khác rỗng. Đây chính là lời giải của bài 236; đúng cho BST nhưng `O(n)` vì không tận dụng thứ tự.

In [13]:
def lca_general_recursive(root, p, q):
    if root is None or root is p or root is q:
        return root
    left = lca_general_recursive(root.left, p, q)
    right = lca_general_recursive(root.right, p, q)
    if left and right:
        return root                                 # p và q ở hai phía
    return left or right


for values, pv, qv, expected in examples:
    root = build_tree(values)
    got = lca_general_recursive(root, find_node(root, pv), find_node(root, qv))
    print(f"root={values}, p={pv}, q={qv} -> {got.val} (kỳ vọng {expected})")

root=[6, 2, 8, 0, 4, 7, 9, None, None, 3, 5], p=2, q=8 -> 6 (kỳ vọng 6)
root=[6, 2, 8, 0, 4, 7, 9, None, None, 3, 5], p=2, q=4 -> 2 (kỳ vọng 2)
root=[2, 1], p=2, q=1 -> 2 (kỳ vọng 2)


**Độ phức tạp Cách 5:** **Time `O(n)`**, **Space `O(h)`**.

> Cách 4 là cách nộp bài chuẩn. Cách 1 và 5 đúng cho mọi cây nhị phân. Cách 2, 3 làm rõ ý "đi theo giá trị".

## 8. TRACE từng bước

Ta theo dõi Cách 4 trên `p = 3`, `q = 5` (cần đi sâu) và `p = 2`, `q = 4` (nút là tổ tiên của chính mình); rồi so sánh với tiền tố chung của hai đường đi.

In [14]:
def trace_lca(values, pv, qv):
    root = build_tree(values)
    p, q = find_node(root, pv), find_node(root, qv)
    lo, hi = min(p.val, q.val), max(p.val, q.val)
    print(f"cây={values}, p={pv}, q={qv}  => lo={lo}, hi={hi}")
    cur = root
    while cur:
        if cur.val < lo:
            print(f"  nút {cur.val} < lo={lo}: cả hai lớn hơn -> sang PHẢI")
            cur = cur.right
        elif cur.val > hi:
            print(f"  nút {cur.val} > hi={hi}: cả hai nhỏ hơn -> sang TRÁI")
            cur = cur.left
        else:
            print(f"  nút {cur.val}: lo={lo} <= {cur.val} <= hi={hi} -> TÁCH RA / trùng nút: LCA = {cur.val}")
            return cur
    return None


tree = [6, 2, 8, 0, 4, 7, 9, None, None, 3, 5]
assert trace_lca(tree, 3, 5).val == 4
print()
assert trace_lca(tree, 2, 4).val == 2
print()
assert trace_lca(tree, 7, 9).val == 8

cây=[6, 2, 8, 0, 4, 7, 9, None, None, 3, 5], p=3, q=5  => lo=3, hi=5
  nút 6 > hi=5: cả hai nhỏ hơn -> sang TRÁI
  nút 2 < lo=3: cả hai lớn hơn -> sang PHẢI
  nút 4: lo=3 <= 4 <= hi=5 -> TÁCH RA / trùng nút: LCA = 4

cây=[6, 2, 8, 0, 4, 7, 9, None, None, 3, 5], p=2, q=4  => lo=2, hi=4
  nút 6 > hi=4: cả hai nhỏ hơn -> sang TRÁI
  nút 2: lo=2 <= 2 <= hi=4 -> TÁCH RA / trùng nút: LCA = 2

cây=[6, 2, 8, 0, 4, 7, 9, None, None, 3, 5], p=7, q=9  => lo=7, hi=9
  nút 6 < lo=7: cả hai lớn hơn -> sang PHẢI
  nút 8: lo=7 <= 8 <= hi=9 -> TÁCH RA / trùng nút: LCA = 8


Bảng tóm tắt (`p = 3`, `q = 5`; `lo = 3`, `hi = 5`):

| nút hiện tại | so sánh | hành động |
|---|---|---|
| `6` | `6 > hi = 5` | sang trái |
| `2` | `2 < lo = 3` | sang phải |
| `4` | `3 <= 4 <= 5` | **trả `4`** |

Đường đi tới `3` là `6 → 2 → 4 → 3`, tới `5` là `6 → 2 → 4 → 5`: tiền tố chung `6, 2, 4`, nút cuối `4` — trùng với kết quả của vòng lặp.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 4 (vòng lặp). Đúng class `Solution`, đúng method `lowestCommonAncestor(self, root, p, q)`.

In [15]:
class Solution:
    def lowestCommonAncestor(self, root: 'TreeNode', p: 'TreeNode', q: 'TreeNode') -> 'TreeNode':
        lo, hi = min(p.val, q.val), max(p.val, q.val)     # p, q có thể ở thứ tự bất kỳ
        while True:
            if root.val < lo:              # cả hai nằm bên phải
                root = root.right
            elif root.val > hi:            # cả hai nằm bên trái
                root = root.left
            else:                          # tách ra, hoặc root trùng p / q
                return root


for values, pv, qv, expected in examples:
    root = build_tree(values)
    got = Solution().lowestCommonAncestor(root, find_node(root, pv), find_node(root, qv))
    assert got.val == expected and got is find_node(root, expected)
print("PASS ví dụ đề cho class Solution")

PASS ví dụ đề cho class Solution


## 10. Vì sao thuật toán đúng

- **Bất biến:** khi đang ở nút `v`, cả `p` và `q` đều nằm trong cây con gốc `v` (ban đầu `v` là gốc nên đúng). LCA của chúng cũng nằm trong cây con của `v`.
- **Trường hợp `v.val < lo`:** `p.val, q.val > v.val` nên theo tính chất BST cả hai nằm ở cây con **phải** của `v` (hoặc không tồn tại; đề đảm bảo tồn tại). Chuyển sang `v.right` giữ nguyên bất biến. `v` không thể là LCA vì cây con phải của `v` đã chứa cả `p` và `q` và nằm **thấp hơn** `v`.
- **Trường hợp `v.val > hi`:** đối xứng với cây con trái.
- **Trường hợp `lo ≤ v.val ≤ hi`:** nếu `lo < v.val < hi` thì `p` ở một phía, `q` ở phía còn lại: không có con nào của `v` chứa cả hai, mà `v` chứa cả hai: `v` là LCA. Nếu `v.val == lo` (hoặc `hi`) thì `v` chính là một trong hai nút, nút còn lại nằm trong cây con của `v` (bất biến), nên `v` là hậu duệ chung và đây là nút sâu nhất có thể (một nút không thể thấp hơn chính nó): `v` là LCA.
- **Kết thúc:** mỗi bước đi xuống một tầng, cây hữu hạn nên dừng sau tối đa `h` bước.
- **Cách 1, 2, 5** đúng theo định nghĩa (tiền tố chung của hai đường đi từ gốc; hoặc hậu thứ tự), và được đối chiếu với oracle độc lập trong phần test.

## 11. Phân tích Time / Space Complexity

Gọi `n` là số nút, `h` là độ cao cây (`log n ≤ h ≤ n`).

- **Cách 1 (đường đi trên cây tổng quát):** Time `O(n)`, Space `O(n)`.
- **Cách 2 (đường đi theo giá trị BST):** Time `O(h)`, Space `O(h)`.
- **Cách 3 (đệ quy theo BST):** Time `O(h)`, Space `O(h)` (ngăn xếp đệ quy).
- **Cách 4 (vòng lặp):** Time `O(h)`, Space `O(1)`.
- **Cách 5 (cây nhị phân tổng quát):** Time `O(n)`, Space `O(h)`.
- BST cân bằng: `h ≈ log n`; BST lệch: `h = n`. Trường hợp xấu nhất của Cách 2–4 là `O(n)`.

## 12. Bộ test

Ta gom các cách vào từ điển `impls`. `oracle` đi **đúng theo định nghĩa**: với mọi nút `c`, tính tập nút trong cây con của `c`; các nút `c` chứa cả `p` và `q` là tổ tiên chung; LCA là tổ tiên chung có cây con **nhỏ nhất**. Cây ngẫu nhiên là BST sinh bằng chèn các giá trị phân biệt.

In [16]:
sol = Solution()
impls = {
    "paths_general": lca_paths_general,
    "paths_bst": lca_paths_bst,
    "recursive": lca_recursive,
    "iterative": lca_iterative,
    "general_recursive": lca_general_recursive,
    "submit": sol.lowestCommonAncestor,
}


def subtree_nodes(node):
    out, stack = set(), [node]
    while stack:
        cur = stack.pop()
        out.add(cur)
        if cur.left:
            stack.append(cur.left)
        if cur.right:
            stack.append(cur.right)
    return out


def oracle(root, p, q):
    """LCA theo định nghĩa: tổ tiên chung có cây con nhỏ nhất."""
    best, best_size = None, None
    stack = [root]
    while stack:
        c = stack.pop()
        sub = subtree_nodes(c)
        if p in sub and q in sub and (best is None or len(sub) < best_size):
            best, best_size = c, len(sub)
        if c.left:
            stack.append(c.left)
        if c.right:
            stack.append(c.right)
    return best


def bst_insert(root, val):
    if root is None:
        return TreeNode(val)
    cur = root
    while True:
        if val < cur.val:
            if cur.left is None:
                cur.left = TreeNode(val)
                return root
            cur = cur.left
        else:
            if cur.right is None:
                cur.right = TreeNode(val)
                return root
            cur = cur.right


def random_bst(n, lo=-10**9, hi=10**9):
    values = random.sample(range(lo, hi + 1), n) if hi - lo < 10**7 else [random.randint(lo, hi) for _ in range(n)]
    values = list(dict.fromkeys(values))              # giá trị phân biệt
    root = None
    for v in values:
        root = bst_insert(root, v)
    return root, values


for values, pv, qv, expected in examples:
    root = build_tree(values)
    p, q = find_node(root, pv), find_node(root, qv)
    assert oracle(root, p, q).val == expected
    for name, fn in impls.items():
        got = fn(root, p, q)
        assert got is oracle(root, p, q), (name, values, pv, qv, got.val)
print("PASS ví dụ của đề (mọi cách và oracle độc lập, so sánh theo danh tính nút)")

PASS ví dụ của đề (mọi cách và oracle độc lập, so sánh theo danh tính nút)


### 12.1. Các trường hợp biên

In [17]:
def check_all(root, p, q, expected_val, label, skip=()):
    for name, fn in impls.items():
        if name in skip:
            continue
        got = fn(root, p, q)
        assert got.val == expected_val, (label, name, got.val, expected_val)
    print("PASS", label, "->", expected_val)


r = build_tree([2, 1])
check_all(r, find_node(r, 2), find_node(r, 1), 2, "hai nút: LCA là gốc (gốc là tổ tiên của chính nó)")
r = build_tree([2, 1])
check_all(r, find_node(r, 1), find_node(r, 2), 2, "đảo thứ tự p, q (p.val < q.val)")
r = build_tree([6, 2, 8, 0, 4, 7, 9, None, None, 3, 5])
check_all(r, find_node(r, 8), find_node(r, 2), 6, "p.val > q.val: lo/hi đảm bảo đúng")
check_all(r, find_node(r, 0), find_node(r, 5), 2, "hai nút ở hai nhánh con của 2")
check_all(r, find_node(r, 3), find_node(r, 5), 4, "hai lá cùng cha")
check_all(r, find_node(r, 7), find_node(r, 9), 8, "hai lá của cây con phải")
check_all(r, find_node(r, 6), find_node(r, 9), 6, "p là gốc")
check_all(r, find_node(r, 4), find_node(r, 3), 4, "q là con của p")
r = build_tree([-10**9, None, 10**9])
check_all(r, find_node(r, -10**9), find_node(r, 10**9), -10**9, "giá trị biên +-10^9")

# Cây lệch phải sâu 3000: các cách đệ quy sẽ lỗi, chỉ kiểm tra các cách lặp
chain = None
for v in range(3000):
    chain = bst_insert(chain, v)
check_all(chain, find_node(chain, 10), find_node(chain, 2999), 10, "cây lệch sâu 3000", skip=("recursive", "general_recursive"))
check_all(chain, find_node(chain, 2998), find_node(chain, 2999), 2998, "cây lệch: hai nút cuối", skip=("recursive", "general_recursive"))
try:
    lca_recursive(chain, find_node(chain, 2998), find_node(chain, 2999))
    print("(đệ quy không gặp lỗi ở giới hạn hiện tại)")
except RecursionError:
    print("lca_recursive: RecursionError trên cây lệch sâu 3000 (vòng lặp Cách 4 không bị)")

PASS hai nút: LCA là gốc (gốc là tổ tiên của chính nó) -> 2
PASS đảo thứ tự p, q (p.val < q.val) -> 2
PASS p.val > q.val: lo/hi đảm bảo đúng -> 6
PASS hai nút ở hai nhánh con của 2 -> 2
PASS hai lá cùng cha -> 4
PASS hai lá của cây con phải -> 8
PASS p là gốc -> 6
PASS q là con của p -> 4
PASS giá trị biên +-10^9 -> -1000000000


PASS cây lệch sâu 3000 -> 10
PASS cây lệch: hai nút cuối -> 2998
lca_recursive: RecursionError trên cây lệch sâu 3000 (vòng lặp Cách 4 không bị)


### 12.2. Tính chất và phản ví dụ

In [18]:
random.seed(235)

# Tính chất 1: LCA(p, q) == LCA(q, p); LCA(p, p-tổ-tiên) là chính tổ tiên; LCA nằm trong [lo, hi]
for _ in range(300):
    n = random.randint(2, 40)
    root, values = random_bst(n)
    pv, qv = random.sample(values, 2)
    p, q = find_node(root, pv), find_node(root, qv)
    a = sol.lowestCommonAncestor(root, p, q)
    b = sol.lowestCommonAncestor(root, q, p)
    assert a is b
    assert min(pv, qv) <= a.val <= max(pv, qv)
print("PASS tính chất: đối xứng và LCA có giá trị nằm giữa p và q")

# Tính chất 2: LCA của một nút và tổ tiên của nó là chính tổ tiên đó
for _ in range(200):
    n = random.randint(2, 40)
    root, values = random_bst(n)
    pv = random.choice(values)
    path = node_path_bst(root, find_node(root, pv))
    anc = random.choice(path)
    assert sol.lowestCommonAncestor(root, find_node(root, pv), anc) is anc if anc.val != pv else True
print("PASS tính chất: LCA(nút, tổ tiên) = tổ tiên")

# Phản ví dụ: giả định p.val < q.val (không sắp lo, hi)
def sai_khong_sap(root, p, q):
    lo, hi = p.val, q.val                       # sai nếu p.val > q.val
    steps = 0
    while root and steps < 100:
        if root.val < lo:
            root = root.right
        elif root.val > hi:
            root = root.left
        else:
            return root
        steps += 1
    return None

r = build_tree([6, 2, 8, 0, 4, 7, 9, None, None, 3, 5])
got = sai_khong_sap(r, find_node(r, 8), find_node(r, 2))
print("p=8, q=2 -> sai:", got.val if got else None, "| đúng:", sol.lowestCommonAncestor(r, find_node(r, 8), find_node(r, 2)).val)
assert got is None or got.val != 6

PASS tính chất: đối xứng và LCA có giá trị nằm giữa p và q
PASS tính chất: LCA(nút, tổ tiên) = tổ tiên
p=8, q=2 -> sai: None | đúng: 6


### 12.3. Đối chiếu ngẫu nhiên (có seed) với oracle độc lập

In [19]:
random.seed(2026)
for trial in range(1500):
    n = random.randint(2, 40)
    root, values = random_bst(n, lo=-100, hi=100) if random.random() < 0.5 else random_bst(n)
    pv, qv = random.sample(values, 2)
    p, q = find_node(root, pv), find_node(root, qv)
    expected = oracle(root, p, q)
    for name, fn in impls.items():
        got = fn(root, p, q)
        assert got is expected, (name, pv, qv, got.val, expected.val)

for trial in range(30):                           # cây lớn: chỉ so với Cách 4 và Cách 1 (oracle O(n^2) quá chậm)
    n = random.randint(500, 1500)
    root, values = random_bst(n)
    pv, qv = random.sample(values, 2)
    p, q = find_node(root, pv), find_node(root, qv)
    assert lca_iterative(root, p, q) is lca_paths_general(root, p, q) is lca_general_recursive(root, p, q)
print("PASS 1530 cây ngẫu nhiên: mọi cách khớp oracle theo định nghĩa (so sánh danh tính nút)")

PASS 1530 cây ngẫu nhiên: mọi cách khớp oracle theo định nghĩa (so sánh danh tính nút)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `or` thay vì `and` khi quyết định đi hướng nào

In [20]:
def sai_or(root, p, q):
    steps = 0
    while root and steps < 100:
        if root.val < p.val or root.val < q.val:        # sai: chỉ cần MỘT trong hai lớn hơn đã sang phải
            root = root.right
        elif root.val > p.val or root.val > q.val:
            root = root.left
        else:
            return root
        steps += 1
    return None


r = build_tree([6, 2, 8, 0, 4, 7, 9, None, None, 3, 5])
got = sai_or(r, find_node(r, 2), find_node(r, 8))
print("p=2, q=8 -> sai:", got.val if got else None, "| đúng:", Solution().lowestCommonAncestor(r, find_node(r, 2), find_node(r, 8)).val)

p=2, q=8 -> sai: None | đúng: 6


**Vì sao sai:** chỉ được đi sang một phía khi **cả hai** nút nằm cùng phía (`and`). Nếu chỉ một nút nằm bên phải thì hai nút đã **tách ra** và nút hiện tại chính là LCA. Dùng `or` làm ta đi quá sâu hoặc ra khỏi cây.

### Lỗi 2: Đòi hỏi `p < nút < q` (bỏ sót trường hợp nút là tổ tiên của chính mình)

In [21]:
def sai_chat(root, p, q):
    lo, hi = min(p.val, q.val), max(p.val, q.val)
    steps = 0
    while root and steps < 100:
        if lo < root.val < hi:                           # sai: dùng < chặt, loại trường hợp bằng
            return root
        root = root.left if root.val > hi else root.right
        steps += 1
    return None


r = build_tree([6, 2, 8, 0, 4, 7, 9, None, None, 3, 5])
got = sai_chat(r, find_node(r, 2), find_node(r, 4))
print("p=2, q=4 -> sai:", got.val if got else None, "| đúng:", Solution().lowestCommonAncestor(r, find_node(r, 2), find_node(r, 4)).val)

p=2, q=4 -> sai: None | đúng: 2


**Vì sao sai:** đề cho phép **một nút là hậu duệ của chính nó**, nên `p = 2` và `q = 4` có LCA là `2` (khi `lo == root.val`). Điều kiện phải là `lo <= root.val <= hi`, tức "không còn cả hai cùng một phía".

### Lỗi 3: Giả định `p.val < q.val`

In [22]:
r = build_tree([6, 2, 8, 0, 4, 7, 9, None, None, 3, 5])
got = sai_khong_sap(r, find_node(r, 8), find_node(r, 2))
print("p=8, q=2 -> sai:", got.val if got else None, "| đúng:", Solution().lowestCommonAncestor(r, find_node(r, 8), find_node(r, 2)).val)

p=8, q=2 -> sai: None | đúng: 6


**Vì sao sai:** đề không đảm bảo `p` nhỏ hơn `q`. Nếu viết `lo, hi = p.val, q.val` thì khi `p.val > q.val` khoảng `[lo, hi]` rỗng và thuật toán đi sai. Luôn dùng `min`/`max`.

### Lỗi 4: Trả về giá trị thay vì nút

In [23]:
def sai_tra_gia_tri(root, p, q):
    return Solution().lowestCommonAncestor(root, p, q).val


r = build_tree([6, 2, 8, 0, 4, 7, 9, None, None, 3, 5])
got = sai_tra_gia_tri(r, find_node(r, 2), find_node(r, 8))
print("kiểu trả về:", type(got).__name__, "| đề cần:", type(find_node(r, 6)).__name__)

kiểu trả về: int | đề cần: TreeNode


**Vì sao sai:** đề yêu cầu trả về **nút** (`TreeNode`), không phải `int`. LeetCode so sánh đối tượng nút trả về với nút đúng; trả giá trị sẽ bị chấm sai. Khi test ở đây ta so sánh danh tính nút (`is`) hoặc `.val` của nút trả về.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**LCA trong BST**": đi từ gốc, so sánh với `min(p.val, q.val)` và `max(p.val, q.val)`; nút đầu tiên nằm trong `[lo, hi]` là đáp án.
- Thấy "LCA trong cây **nhị phân thường**": không có thứ tự giá trị, dùng đệ quy hậu thứ tự hoặc đường đi + bản đồ cha (bài 236).
- Muốn tiết kiệm bộ nhớ và tránh `RecursionError`: viết bằng vòng lặp `while`.
- "Nút là hậu duệ của chính nó": đừng dùng so sánh chặt khi kiểm tra "ở giữa".
- Khi test cây: viết `build_tree`, `find_node` (lấy nút theo giá trị), và một oracle theo **định nghĩa** (tổ tiên chung có cây con nhỏ nhất).
- Đường đi gốc → nút là công cụ vạn năng: mọi khái niệm "tổ tiên" nằm trên đường này.

## 15. Biến thể của bài

1. **LCA của cây nhị phân tổng quát** (LeetCode 236, bài kế tiếp).
2. **Khoảng cách giữa hai nút** trong BST: `depth(p) + depth(q) − 2·depth(LCA)`.
3. **LCA của nhiều nút** (một tập nút): LCA của `min` và `max` giá trị trong BST.
4. **`p` hoặc `q` có thể không có trong cây:** cần kiểm tra tồn tại.
5. **Đường đi giữa hai nút** (từ `p` lên LCA rồi xuống `q`).

In [24]:
def depth_in_bst(root, node):
    d, cur = 0, root
    while cur is not node:
        cur = cur.left if node.val < cur.val else cur.right
        d += 1
    return d


def distance_between(root, p, q):
    """Biến thể 2."""
    lca = lca_iterative(root, p, q)
    return depth_in_bst(root, p) + depth_in_bst(root, q) - 2 * depth_in_bst(root, lca)


def lca_of_many(root, nodes):
    """Biến thể 3: LCA của một tập nút = LCA của nút nhỏ nhất và lớn nhất."""
    lo = min(nodes, key=lambda x: x.val)
    hi = max(nodes, key=lambda x: x.val)
    return lca_iterative(root, lo, hi)


def lca_checked(root, pv, qv):
    """Biến thể 4: trả None nếu một trong hai giá trị không có trong cây."""
    p, q = find_node(root, pv), find_node(root, qv)
    if p is None or q is None:
        return None
    return lca_iterative(root, p, q)


r = build_tree([6, 2, 8, 0, 4, 7, 9, None, None, 3, 5])
assert distance_between(r, find_node(r, 3), find_node(r, 5)) == 2
assert distance_between(r, find_node(r, 0), find_node(r, 9)) == 4
assert lca_of_many(r, [find_node(r, v) for v in (3, 5, 0)]).val == 2
assert lca_checked(r, 3, 99) is None and lca_checked(r, 3, 5).val == 4
random.seed(15)
for _ in range(200):
    root, values = random_bst(random.randint(2, 30), lo=-50, hi=50)
    subset = random.sample(values, random.randint(2, min(5, len(values))))
    nodes = [find_node(root, v) for v in subset]
    ans = lca_of_many(root, nodes)
    expected = nodes[0]
    for x in nodes[1:]:
        expected = oracle(root, expected, x)
    assert ans is expected
print("PASS biến thể: khoảng cách, nhiều nút, nút vắng mặt")

PASS biến thể: khoảng cách, nhiều nút, nút vắng mặt


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đường đi trên cây tổng quát | bản đồ cha, tiền tố chung | `O(n)` | `O(n)` | đúng cho mọi cây; oracle |
| 2. Đường đi theo giá trị BST | đi xuống theo `<`/`>` rồi tiền tố chung | `O(h)` | `O(h)` | dễ hiểu từ định nghĩa |
| 3. Đệ quy theo BST | `lo`, `hi` quyết định hướng | `O(h)` | `O(h)` | gọn, cây không quá sâu |
| 4. Vòng lặp `while` | như Cách 3 không đệ quy | `O(h)` | `O(1)` | **cách nộp bài** |
| 5. Lời giải cây tổng quát | hậu thứ tự, hai bên cùng khác rỗng | `O(n)` | `O(h)` | khi quên tính chất BST / bài 236 |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `min(8, 2)`, `max(8, 2)`, `bst_path(build_tree([4, 2, 6]), 6)`, và LCA (giá trị) của `2` và `6` trong cây `[4, 2, 6]`, LCA của `2` và `4`.

**Bài 2 (tính tay).** Cây `[6,2,8,0,4,7,9,null,null,3,5]`: tìm LCA của `(0, 5)`, `(7, 9)`, `(3, 8)`, `(4, 4)` nếu cho phép `p = q`. Vẽ đường đi từ gốc mỗi nút để kiểm tra.

**Bài 3 (điền TODO).** Hoàn thành `lca_todo(root, p, q)` theo Cách 4 (vòng lặp).

In [25]:
def lca_todo(root, p, q):
    # TODO 1: lo = min(p.val, q.val), hi = max(p.val, q.val)
    # TODO 2: lặp trong khi root còn:
    #   nếu root.val < lo: sang phải; nếu root.val > hi: sang trái; ngược lại trả root
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `lca_paths2(root, p, q)` theo Cách 2 (đường đi theo giá trị, tiền tố chung). Kiểm tra bằng ba ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `distance2(root, p, q)` trả về số cạnh trên đường đi ngắn nhất giữa `p` và `q`. (b) Viết `lca_many2(root, nodes)` trả về LCA của một danh sách nút bất kỳ (ít nhất 2 nút).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `min` và `max` chọn giá trị nhỏ/lớn; đường đi từ gốc `4` tới `6` là `[4, 6]`. Bài 2: dùng quy tắc "tách ra hoặc trùng nút".
- **Mức 2 (vừa):** Bài 1: `2`, `8`, `[4, 6]`, `4`, `4`. Bài 2: `(0,5)`: `6 > 5 → trái → 2`, `2` nằm giữa `0` và `5`: LCA `2`; `(7,9)`: `8`; `(3,8)`: `6`; `(4,4)`: `4`. Bài 3: hai TODO trong comment.
- **Mức 3 (sâu):** Bài 4: dùng `node_path_bst` rồi `zip` hai đường, cập nhật `lca = a` khi `a is b`. Bài 5(a): `depth(p) + depth(q) − 2·depth(lca)` với `depth` là số bước đi từ gốc xuống nút theo giá trị. Bài 5(b): LCA của nút nhỏ nhất và nút lớn nhất theo `val`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [26]:
# Bài 1
assert min(8, 2) == 2 and max(8, 2) == 8
assert bst_path(build_tree([4, 2, 6]), 6) == [4, 6]
r1 = build_tree([4, 2, 6])
assert Solution().lowestCommonAncestor(r1, find_node(r1, 2), find_node(r1, 6)).val == 4
assert Solution().lowestCommonAncestor(r1, find_node(r1, 2), find_node(r1, 4)).val == 4

# Bài 2
r2 = build_tree([6, 2, 8, 0, 4, 7, 9, None, None, 3, 5])
for pv, qv, expected in [(0, 5, 2), (7, 9, 8), (3, 8, 6)]:
    assert Solution().lowestCommonAncestor(r2, find_node(r2, pv), find_node(r2, qv)).val == expected
assert lca_iterative(r2, find_node(r2, 4), find_node(r2, 4)).val == 4

# Bài 3
def lca_todo_ans(root, p, q):
    lo, hi = min(p.val, q.val), max(p.val, q.val)
    while root:
        if root.val < lo:
            root = root.right
        elif root.val > hi:
            root = root.left
        else:
            return root

for values, pv, qv, expected in examples:
    rt = build_tree(values)
    assert lca_todo_ans(rt, find_node(rt, pv), find_node(rt, qv)).val == expected

# Bài 4
def lca_paths2(root, p, q):
    path_p, path_q = node_path_bst(root, p), node_path_bst(root, q)
    lca = None
    for a, b in zip(path_p, path_q):
        if a is not b:
            break
        lca = a
    return lca

for values, pv, qv, expected in examples:
    rt = build_tree(values)
    assert lca_paths2(rt, find_node(rt, pv), find_node(rt, qv)).val == expected

# Bài 5(a), 5(b)
def distance2(root, p, q):
    lca = lca_iterative(root, p, q)
    return depth_in_bst(root, p) + depth_in_bst(root, q) - 2 * depth_in_bst(root, lca)

def lca_many2(root, nodes):
    return lca_iterative(root, min(nodes, key=lambda x: x.val), max(nodes, key=lambda x: x.val))

assert distance2(r2, find_node(r2, 0), find_node(r2, 9)) == 4
assert lca_many2(r2, [find_node(r2, v) for v in (0, 3, 7)]).val == 6
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (LCA trong BST bằng vòng lặp)

```python
class Solution:
    def lowestCommonAncestor(self, root: 'TreeNode', p: 'TreeNode', q: 'TreeNode') -> 'TreeNode':
        lo, hi = min(p.val, q.val), max(p.val, q.val)
        while True:
            if root.val < lo:
                root = root.right
            elif root.val > hi:
                root = root.left
            else:
                return root
```

**Checklist**

- [ ] Dùng tính chất BST: cả hai nhỏ hơn thì sang trái, cả hai lớn hơn thì sang phải, còn lại trả nút hiện tại?
- [ ] Dùng `min`/`max` (không giả định `p.val < q.val`)?
- [ ] Điều kiện dừng là `lo <= root.val <= hi` (không chặt), vì một nút là hậu duệ của chính nó?
- [ ] Trả về **nút** (`TreeNode`), không trả `.val`?
- [ ] Cân nhắc cây lệch (n tới `10⁵`): ưu tiên vòng lặp thay vì đệ quy?
- [ ] Đã thử: hai nút, `p` là tổ tiên của `q`, `p`/`q` đảo thứ tự, hai lá cùng cha?
- [ ] Đã đối chiếu với oracle theo định nghĩa (tổ tiên chung có cây con nhỏ nhất) trên cây ngẫu nhiên?
- [ ] Skeleton đúng `class Solution` / `def lowestCommonAncestor(self, root: 'TreeNode', p: 'TreeNode', q: 'TreeNode') -> 'TreeNode'`?